In [4]:
%pip install transformers datasets accelerate evaluate

Note: you may need to restart the kernel to use updated packages.


In [5]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report
)

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found.")

print("Libraries loaded successfully.")

True
NVIDIA GeForce RTX 3050 Laptop GPU
Libraries loaded successfully.


In [6]:
# LOAD CLEANED DATA

TRAIN_PATH = "../data/interim/train_clean_v2.csv"

train_df = pd.read_csv(TRAIN_PATH)

print(train_df.shape)

display(train_df.head())

(5000, 2)


,full_text,label
0,pret. di sekolah saya dapat mbg tetap saja ...,Sasaran Penerima
1,mbg bentuk penggarongan duit negara secara tsm...,Politik
2,pasal 34 ayat (1) undang-undang dasar negara r...,Sasaran Penerima
3,makan bergizi gratis bikin masyarakat merasa ...,Sasaran Penerima
4,"presiden ngotot, paling kesal kalau mbg dikri...",Politik


In [7]:
# LABEL ENCODING

labels = sorted(train_df['label'].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

train_df['label_id'] = train_df['label'].map(label2id)

print(label2id)

{'Anggaran': 0, 'Distribusi': 1, 'Ekonomi': 2, 'Kualitas Pangan': 3, 'Lainnya': 4, 'Politik': 5, 'Sasaran Penerima': 6, 'Tata Kelola': 7}


In [8]:
# LOAD TOKENIZER

MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")

Tokenizer loaded.


In [9]:
# TOKENIZATION FUNCTION

MAX_LENGTH = 256


def tokenize_function(texts):
    
    return tokenizer(
        
        texts.tolist(),
        
        padding='max_length',
        
        truncation=True,
        
        max_length=MAX_LENGTH
    )

In [10]:
# STRATIFIED KFOLD

skf = StratifiedKFold(
    
    n_splits=5,
    
    shuffle=True,
    
    random_state=42
)

print("StratifiedKFold initialized.")

StratifiedKFold initialized.


In [11]:
# CUSTOM DATASET

from torch.utils.data import Dataset


class TextDataset(Dataset):
    
    def __init__(
        self,
        encodings,
        labels
    ):
        
        self.encodings = encodings
        self.labels = labels
    
    
    def __getitem__(self, idx):
        
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        
        item['labels'] = torch.tensor(
            self.labels[idx]
        )
        
        return item
    
    
    def __len__(self):
        
        return len(self.labels)

In [12]:
# METRIC FUNCTION

import evaluate

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    
    logits, labels = eval_pred
    
    predictions = np.argmax(logits, axis=-1)
    
    
    # balanced accuracy
    balanced_acc = balanced_accuracy_score(
        labels,
        predictions
    )
    
    
    return {
        "balanced_accuracy": balanced_acc
    }

In [13]:
# CROSS VALIDATION TRAINING

fold_scores = []

oof_predictions = np.zeros(len(train_df))

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(
        train_df['full_text'],
        train_df['label_id']
    )
):
    
    print("=" * 60)
    print(f"FOLD {fold + 1}")
    print("=" * 60)
    
        # SPLIT DATA    
    train_fold = train_df.iloc[train_idx]
    valid_fold = train_df.iloc[valid_idx]
    
        # TOKENIZATION    
    train_encodings = tokenize_function(
        train_fold['full_text']
    )
    
    valid_encodings = tokenize_function(
        valid_fold['full_text']
    )
    
        # DATASET    
    train_dataset = TextDataset(
        train_encodings,
        train_fold['label_id'].tolist()
    )
    
    valid_dataset = TextDataset(
        valid_encodings,
        valid_fold['label_id'].tolist()
    )
    
        # MODEL    
    model = AutoModelForSequenceClassification.from_pretrained(
        
        MODEL_NAME,
        
        num_labels=len(label2id),
        
        id2label=id2label,
        
        label2id=label2id
    )
    
        # TRAINING ARGUMENTS    
    training_args = TrainingArguments(
        
        output_dir=f"./fold_{fold+1}",
        
        eval_strategy="epoch",
        
        save_strategy="epoch",
        
        logging_strategy="epoch",
        
        num_train_epochs=2,
        
        per_device_train_batch_size=8,
        
        per_device_eval_batch_size=8,
        
        learning_rate=2e-5,
        
        weight_decay=0.01,
        
        load_best_model_at_end=True,
        
        metric_for_best_model="balanced_accuracy",
        
        greater_is_better=True,
        
        report_to="none"
    )
    
        # TRAINER    
    trainer = Trainer(
        
        model=model,
        
        args=training_args,
        
        train_dataset=train_dataset,
        
        eval_dataset=valid_dataset,
        
        compute_metrics=compute_metrics
    )
    
        # TRAIN    
    trainer.train()
    
        # EVALUATION    
    predictions_output = trainer.predict(
        valid_dataset
    )
    
    predictions = np.argmax(
        predictions_output.predictions,
        axis=-1
    )
    
    
    # save OOF prediction
    oof_predictions[valid_idx] = predictions
    
    
    # score
    score = balanced_accuracy_score(
        valid_fold['label_id'],
        predictions
    )
    
    fold_scores.append(score)
    
    
    print(f"Balanced Accuracy : {score:.4f}")
    print()

FOLD 1


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,1.230700,1.122406,0.622295
2,0.702300,1.064784,0.664353


Balanced Accuracy : 0.6644

FOLD 2


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,1.271000,1.068031,0.612463
2,0.745100,1.021898,0.658976


Balanced Accuracy : 0.6590

FOLD 5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Balanced Accuracy
1,1.280900,1.035125,0.611527
2,0.736500,1.029021,0.635213


Balanced Accuracy : 0.6352



In [15]:
# OVERALL RESULT

print("=" * 60)
print("TRANSFORMER RESULT")
print("=" * 60)

for i, score in enumerate(fold_scores):
    
    print(f"Fold {i+1}: {score:.4f}")

print()

print(f"Mean Balanced Accuracy: {np.mean(fold_scores):.4f}")

print(f"Std Balanced Accuracy : {np.std(fold_scores):.4f}")

TRANSFORMER RESULT
Fold 1: 0.6644
Fold 2: 0.6659
Fold 3: 0.6882
Fold 4: 0.6590
Fold 5: 0.6352

Mean Balanced Accuracy: 0.6625
Std Balanced Accuracy : 0.0169
